In [ ]:

# --- 모델 및 데이터셋 정의 ------------------------------------------------------------------------------------------------------------------------
import time

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import k_hop_subgraph
from torch_geometric.nn import SAGEConv, global_mean_pool
from sklearn.metrics import roc_auc_score, average_precision_score

def drnl_node_labeling(adj, src, dst, simple_version=False):
    # Double Radius Node Labeling (DRNL).
    if not simple_version:
        src, dst = (dst, src) if src > dst else (src, dst)

        idx = list(range(src)) + list(range(src + 1, adj.shape[0]))
        adj_wo_src = adj[idx, :][:, idx]

        idx = list(range(dst)) + list(range(dst + 1, adj.shape[0]))
        adj_wo_dst = adj[idx, :][:, idx]

        dist2src = shortest_path(adj_wo_dst, directed=False, unweighted=True, indices=src)
        dist2src = np.insert(dist2src, dst, 0, axis=0)
        dist2src = torch.from_numpy(dist2src)

        dist2dst = shortest_path(adj_wo_src, directed=False, unweighted=True, indices=dst-1)
        dist2dst = np.insert(dist2dst, src, 0, axis=0)
        dist2dst = torch.from_numpy(dist2dst)

        dist = dist2src + dist2dst
        dist_over_2, dist_mod_2 = dist // 2, dist % 2

        z = 1 + torch.min(dist2src, dist2dst)
        z += dist_over_2 * (dist_over_2 + dist_mod_2 - 1)
        z[src] = 1.
        z[dst] = 1.
        z[torch.isnan(z)] = 0.
    else:
        # 간단한 구조 특징 추가: 소스/타깃 노드엔 1.0, 주변 노드엔 0.0 부여 (DRNL의 간소화 버전)
        z = torch.zeros((adj.shape[0], 1), dtype=torch.float)
        z[src] = 1.0
        z[dst] = 1.0

    return z.to(torch.long)

class SEALSubgraphDataset(torch.utils.data.Dataset):
    def __init__(self, full_data, link_index, link_labels, num_hops=2):
        '''
        full_data: 원본 데이터
        link_index: 링크 쌍 [2, num_edges]
        link_labels: 1 = 링크 존재 0 = 링크  존재 x
        '''
        self.x = full_data.x
        self.edge_index = full_data.edge_index
        self.links = link_index.t().tolist()
        self.labels = link_labels.tolist()
        self.num_hops = num_hops

    def __len__(self):
        return len(self.links)
    
    def __getitem__(self, idx):
        src, dst = self.links[idx]
        y = self.labels[idx]

        # 두 노드 주변의 k-hop 서브 그래프 추출
        subnodes, subedges, mapping, edge_mask = k_hop_subgraph(
            node_idx=[src, dst],
            num_hops=self.num_hops,
            edge_index=self.edge_index,
            relabel_nodes=True
        )

        # 서브그래프에 속한 노드들의 원래 Feature 추출
        sub_x = self.x[subnodes]

        num_nodes = len(subnodes)
        row = subedges[0]
        col = subedges[1]
        weight = np.ones(subedges.shape[1], dtype=int)
        adj = csr_matrix((weight, (row, col)), shape=(num_nodes, num_nodes))

        # 원래의 소스(src), 타깃( dst) 노드가 서브 그래프 안에서 몇 번 인덱스로 바뀌었는지 확인
        # 이 mapping 정보를 통해 DRNL 등 구조적 라벨링을 추가할 수 있음.
        src_new, dst_new = mapping[0].item(), mapping[1].item()

        # 간단한 구조 특징 추가: 소스/타깃 노드엔 1.0, 주변 노드엔 0.0 부여 (DRNL의 간소화 버전)
        structural_feat = torch.zeros((sub_x.size(0), 1), dtype=torch.float)
        structural_feat[src_new] = 1.0
        structural_feat[dst_new] = 1.0
        z = drnl_node_labeling(adj, src_new, dst_new, simple_version=False)
        z = z.reshape(-1, 1)

        final_x = torch.cat([sub_x, structural_feat], dim=-1)

        # PyG의 Data 객체와 유사하게
        return {
            'x': final_x,
            'edge_index': subedges,
            'y': torch.tensor(y, dtype=torch.float)
        }
    
def collate_fn(batch):
    # 여러 서브 그래프를 하나의 그래프로 합침.
    num_nodes_cum = 0
    batch_x = []
    batch_edge_index = []
    batch_y = []
    batch_index = []
    
    for i, data in enumerate(batch):
        x, edge_index, y = data['x'], data['edge_index'], data['y']
        num_nodes = x.size(0)

        batch_x.append(x)
        batch_edge_index.append(edge_index + num_nodes_cum)
        batch_y.append(y)
        # 어떤 노드가 몇 번째 서브그래프(배치)에 속하는 기록
        batch_index.append(torch.full((num_nodes,), i, dtype=torch.long))

        num_nodes_cum += num_nodes

    print({'x': torch.cat(batch_x, dim=0),
        'edge_index': torch.cat(batch_edge_index, dim=1),
        'y': torch.stack(batch_y, dim=0), 
        'batch': torch.cat(batch_index, dim=0)
    })
    return {
        'x': torch.cat(batch_x, dim=0),
        'edge_index': torch.cat(batch_edge_index, dim=1),
        'y': torch.stack(batch_y, dim=0), 
        'batch': torch.cat(batch_index, dim=0)
    }


class SEALClassifier(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.lin1 = torch.nn.Linear(hidden_channels, 64)
        self.lin2 = torch.nn.Linear(64, 1)

    def forward(self, x, edge_index, batch):
        # 1. 서브 그래프 내부 노드 전파
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))

        # 2. Graph-level Pooling (각 서브그래프별로 하나의 벡터 추출)
        h = global_mean_pool(h, batch)

        # 3. 최종 분류 
        h = F.relu(self.lin1(h))
        h = F.dropout(h, p=0.5, training=self.training)
        return torch.sigmoid(self.lin2(h)).squeeze(-1)


def train(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch['x'].to(device), batch['edge_index'].to(device), batch['batch'].to(device))
        loss = criterion(out, batch['y'].to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds, targets = [], []
    for batch in loader:
        out = model(batch['x'].to(device), batch['edge_index'].to(device), batch['batch'].to(device))
        preds.extend(out.cpu().tolist())
        targets.extend(batch['y'].tolist())
    return  roc_auc_score(targets, preds), average_precision_score(targets, preds)



In [ ]:
# - Cora Dataset test---
dataset = Planetoid(root='./data/Cora', name="Cora")
data = dataset[0]

transform = RandomLinkSplit(num_val=0.1, num_test=0.1, is_undirected=True, add_negative_train_samples=True)
train_data, val_data, test_data = transform(data)

train_dataset = SEALSubgraphDataset(train_data, train_data.edge_label_index, train_data.edge_label)
val_dataset = SEALSubgraphDataset(val_data, val_data.edge_label_index, val_data.edge_label)
test_dataset = SEALSubgraphDataset(test_data, test_data.edge_label_index, test_data.edge_label)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'---dataset is Cora----')
print(f'---device is {device} ---')
model = SEALClassifier(in_channels=dataset.num_features + 1, hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.BCELoss()

print('------학습시작-------')
start_time = time.time()
for epoch in range(1, 11):
    loss = train(model, train_loader, optimizer, criterion, device)
    val_auc, val_ap = evaluate(model, val_loader, device)
    print(f"Epoch: {epoch:02d}, Loss: {loss:.4f}, Val AUC: {val_auc:.4f}, Val AP: {val_ap:.4f}")

print(f"--------학습 완료--------")

test_auc, test_ap = evaluate(model, test_loader, device)
print(f"--------Cora: Test AUC: {test_auc:.4f}, Test AP: {test_ap:.4f}")
print(f"--------총 걸린시간(s): {time.time() - start_time}")

c:\Users\pmw9440\AppData\Local\pypoetry\Cache\virtualenvs\seal-5OwgjiJG-py3.11\Lib\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feat

---dataset is Cora----
---device is cuda ---
------학습시작-------
Epoch: 01, Loss: 0.6940, Val AUC: 0.5558, Val AP: 0.5646
Epoch: 02, Loss: 0.6851, Val AUC: 0.5782, Val AP: 0.6113
Epoch: 03, Loss: 0.6662, Val AUC: 0.6034, Val AP: 0.6013
Epoch: 04, Loss: 0.6575, Val AUC: 0.6227, Val AP: 0.6207
Epoch: 05, Loss: 0.6417, Val AUC: 0.6210, Val AP: 0.6082
Epoch: 06, Loss: 0.6039, Val AUC: 0.6262, Val AP: 0.6288
Epoch: 07, Loss: 0.2791, Val AUC: 0.5885, Val AP: 0.5962
Epoch: 08, Loss: 0.1016, Val AUC: 0.5659, Val AP: 0.5751
Epoch: 09, Loss: 0.0535, Val AUC: 0.5680, Val AP: 0.5745
Epoch: 10, Loss: 0.0466, Val AUC: 0.5653, Val AP: 0.5699
--------학습 완료--------
--------Cora: Test AUC: 0.5837, Test AP: 0.5875
--------총 걸린시간(s): 143.96742391586304


In [4]:
# - Citeseer Dataset test---
dataset = Planetoid(root='./data/Citeseer', name="Citeseer")
data = dataset[0]

transform = RandomLinkSplit(num_val=0.1, num_test=0.1, is_undirected=True, add_negative_train_samples=True)
train_data, val_data, test_data = transform(data)

train_dataset = SEALSubgraphDataset(train_data, train_data.edge_label_index, train_data.edge_label)
val_dataset = SEALSubgraphDataset(val_data, val_data.edge_label_index, val_data.edge_label)
test_dataset = SEALSubgraphDataset(test_data, test_data.edge_label_index, test_data.edge_label)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'---dataset is Citeseer----')
print(f'---device is {device} ---')
model = SEALClassifier(in_channels=dataset.num_features + 1, hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.BCELoss()

print('------학습시작-------')
start_time = time.time()
for epoch in range(1, 11):
    loss = train(model, train_loader, optimizer, criterion, device)
    val_auc, val_ap = evaluate(model, val_loader, device)
    print(f"Epoch: {epoch:02d}, Loss: {loss:.4f}, Val AUC: {val_auc:.4f}, Val AP: {val_ap:.4f}")

print(f"--------학습 완료--------")

test_auc, test_ap = evaluate(model, test_loader, device)
print(f"--------Citeseer: Val AUC: {test_auc:.4f}, Val AP: {test_ap:.4f}")
print(f"--------총 걸린시간(s): {time.time() - start_time}")

---dataset is Citeseer----
---device is cuda ---
------학습시작-------
Epoch: 01, Loss: 0.6824, Val AUC: 0.5864, Val AP: 0.6152
Epoch: 02, Loss: 0.5700, Val AUC: 0.5967, Val AP: 0.6524
Epoch: 03, Loss: 0.3112, Val AUC: 0.5818, Val AP: 0.6294
Epoch: 04, Loss: 0.1214, Val AUC: 0.5827, Val AP: 0.6492
Epoch: 05, Loss: 0.0908, Val AUC: 0.5814, Val AP: 0.6526
Epoch: 06, Loss: 0.0270, Val AUC: 0.5832, Val AP: 0.6401
Epoch: 07, Loss: 0.0340, Val AUC: 0.5862, Val AP: 0.6314
Epoch: 08, Loss: 0.0438, Val AUC: 0.5889, Val AP: 0.6501
Epoch: 09, Loss: 0.0397, Val AUC: 0.5863, Val AP: 0.6500
Epoch: 10, Loss: 0.0229, Val AUC: 0.5930, Val AP: 0.6551
--------학습 완료--------
--------Citeseer: Val AUC: 0.5841, Val AP: 0.6551
--------총 걸린시간(s): 132.52045154571533


In [5]:
# - Pubmed Dataset test---
dataset = Planetoid(root='./data/Pubmed', name="Pubmed")
data = dataset[0]

transform = RandomLinkSplit(num_val=0.1, num_test=0.1, is_undirected=True, add_negative_train_samples=True)
train_data, val_data, test_data = transform(data)

train_dataset = SEALSubgraphDataset(train_data, train_data.edge_label_index, train_data.edge_label)
val_dataset = SEALSubgraphDataset(val_data, val_data.edge_label_index, val_data.edge_label)
test_dataset = SEALSubgraphDataset(test_data, test_data.edge_label_index, test_data.edge_label)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'---dataset is Pubmed----')
print(f'---device is {device} ---')
model = SEALClassifier(in_channels=dataset.num_features + 1, hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.BCELoss()

print('------학습시작-------')
start_time = time.time()
for epoch in range(1, 11):
    loss = train(model, train_loader, optimizer, criterion, device)
    val_auc, val_ap = evaluate(model, val_loader, device)
    print(f"Epoch: {epoch:02d}, Loss: {loss:.4f}, Val AUC: {val_auc:.4f}, Val AP: {val_ap:.4f}")

print(f"--------학습 완료--------")

test_auc, test_ap = evaluate(model, test_loader, device)
print(f"--------Pubmed: Val AUC: {test_auc:.4f}, Val AP: {test_ap:.4f}")
print(f"--------총 걸린시간(s): {time.time() - start_time}")

Processing...
Done!
c:\Users\pmw9440\AppData\Local\pypoetry\Cache\virtualenvs\seal-5OwgjiJG-py3.11\Lib\site-packages\torch_geometric\io\fs.py:215: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this expe

---dataset is Pubmed----
---device is cuda ---
------학습시작-------


IndexError: index 19716 is out of bounds for dimension 0 with size 19715

In [ ]:
import torch
from scipy.sparse import csr_matrix
edge_list = np.array([[0, 1, 2, 3, 4, 5],
                     [ 2, 2, 4, 4, 6, 6]])

num_nodes = 7
row = edge_list[0]
col = edge_list[1]

data = np.ones(edge_list.shape[1], dtype=int)
adj = csr_matrix((data, (row, col)), shape=(num_nodes, num_nodes))

src, dst = 0, 1

z = drnl_node_labeling(adj, src, dst)
print(z)

NameError: name 'np' is not defined

In [ ]:
adj.shape[0]

7